# Data Processing - INX Future Inc Employee Performance Analysis

## Objective
This notebook performs data loading, cleaning, and preprocessing for the employee performance analysis.

## Steps
1. Load raw data
2. Data quality assessment
3. Handle missing values
4. Data type corrections
5. Feature engineering
6. Save processed data

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
import pandas as pd

# File path
file_path = "data/raw/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1.8.xls"

# If running in this environment use:
# file_path = "/mnt/data/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1.8 (1).xls"

# Load dataset
df = pd.read_excel(file_path, engine="xlrd")

# Extract column names
columns = df.columns.tolist()

# Save to txt
with open("column_names.txt", "w") as f:
    for col in columns:
        f.write(col + "\n")

print("Column names saved to column_names.txt")

Column names saved to column_names.txt


## 1. Load Raw Data

In [2]:
# Load the dataset
data_path = '../../data/raw/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1.8.xls'

# Try different engines for Excel file reading
try:
    df = pd.read_excel(data_path, engine='xlrd')
except:
    try:
        df = pd.read_excel(data_path, engine='openpyxl')
    except:
        print("Error loading file. Please check the file path and format.")
        df = None

if df is not None:
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"\nFirst few rows:")
    display(df.head())

Error loading file. Please check the file path and format.


## 2. Data Quality Assessment

In [3]:
# Basic information
print("=" * 80)
print("DATA INFORMATION")
print("=" * 80)
df.info()

DATA INFORMATION


AttributeError: 'NoneType' object has no attribute 'info'

In [ ]:
# Statistical summary
print("\n" + "=" * 80)
print("STATISTICAL SUMMARY")
print("=" * 80)
display(df.describe())

In [ ]:
# Check for missing values
print("\n" + "=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_data) > 0:
    display(missing_data)
else:
    print("No missing values found!")

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

if duplicates > 0:
    print("Removing duplicate rows...")
    df = df.drop_duplicates()
    print(f"New shape after removing duplicates: {df.shape}")

## 3. Data Type Analysis and Corrections

In [ ]:
# Identify column types
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Check unique values for categorical columns
print("\n" + "=" * 80)
print("CATEGORICAL VARIABLES - UNIQUE VALUES")
print("=" * 80)

for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f"\n{col}: {unique_count} unique values")
    if unique_count <= 20:
        print(df[col].value_counts())
    else:
        print(f"Too many unique values. Showing top 10:")
        print(df[col].value_counts().head(10))

## 4. Handle Missing Values

In [ ]:
# Strategy for handling missing values
# Numerical: Median imputation or mean (depending on distribution)
# Categorical: Mode imputation or 'Unknown' category

from sklearn.impute import SimpleImputer

# For numerical columns with missing values
if len(missing_data[missing_data['Column'].isin(numerical_cols)]) > 0:
    print("Imputing numerical columns with median...")
    num_imputer = SimpleImputer(strategy='median')
    
    for col in numerical_cols:
        if df[col].isnull().sum() > 0:
            df[col] = num_imputer.fit_transform(df[[col]])
            print(f"  - {col}: Imputed with median")

# For categorical columns with missing values
if len(missing_data[missing_data['Column'].isin(categorical_cols)]) > 0:
    print("\nImputing categorical columns with mode...")
    
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            mode_value = df[col].mode()[0]
            df[col].fillna(mode_value, inplace=True)
            print(f"  - {col}: Imputed with mode ({mode_value})")

# Verify no missing values remain
print(f"\nTotal missing values after imputation: {df.isnull().sum().sum()}")

## 5. Feature Engineering

In [ ]:
# Create a copy for feature engineering
df_processed = df.copy()

print("Feature Engineering Steps:")
print("=" * 80)

# Example feature engineering (adjust based on actual columns)
# These are common transformations for HR data

# 1. Age groups
if 'Age' in df_processed.columns:
    df_processed['Age_Group'] = pd.cut(df_processed['Age'], 
                                       bins=[0, 25, 35, 45, 55, 100],
                                       labels=['<25', '25-35', '35-45', '45-55', '55+'])
    print("✓ Created Age_Group feature")

# 2. Tenure groups
if 'ExperienceYearsAtThisCompany' in df_processed.columns:
    df_processed['Tenure_Group'] = pd.cut(df_processed['ExperienceYearsAtThisCompany'],
                                          bins=[0, 2, 5, 10, 100],
                                          labels=['0-2 years', '2-5 years', '5-10 years', '10+ years'])
    print("✓ Created Tenure_Group feature")

# 3. Salary categories
if 'MonthlyIncome' in df_processed.columns:
    df_processed['Salary_Category'] = pd.qcut(df_processed['MonthlyIncome'],
                                              q=4,
                                              labels=['Low', 'Medium', 'High', 'Very High'])
    print("✓ Created Salary_Category feature")

# 4. Years since promotion ratio
if 'YearsSinceLastPromotion' in df_processed.columns and 'ExperienceYearsAtThisCompany' in df_processed.columns:
    df_processed['Promotion_Ratio'] = (df_processed['YearsSinceLastPromotion'] / 
                                       (df_processed['ExperienceYearsAtThisCompany'] + 1)).round(3)
    print("✓ Created Promotion_Ratio feature")

# 5. Job satisfaction composite
satisfaction_cols = [col for col in df_processed.columns if 'Satisfaction' in col]
if len(satisfaction_cols) > 0:
    df_processed['Average_Satisfaction'] = df_processed[satisfaction_cols].mean(axis=1).round(2)
    print(f"✓ Created Average_Satisfaction from {len(satisfaction_cols)} satisfaction metrics")

print(f"\nProcessed dataset shape: {df_processed.shape}")
print(f"New features created: {df_processed.shape[1] - df.shape[1]}")

## 6. Outlier Detection and Treatment

In [ ]:
# Detect outliers using IQR method for numerical columns
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print("Outlier Analysis (IQR Method):")
print("=" * 80)

outlier_summary = []
for col in numerical_cols:
    if col != 'EmpNumber':  # Skip ID column
        n_outliers, lower, upper = detect_outliers_iqr(df_processed, col)
        if n_outliers > 0:
            outlier_summary.append({
                'Column': col,
                'Outliers': n_outliers,
                'Lower_Bound': round(lower, 2),
                'Upper_Bound': round(upper, 2)
            })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
    display(outlier_df)
else:
    print("No significant outliers detected.")

# Note: Decide whether to remove, cap, or keep outliers based on domain knowledge

## 7. Save Processed Data

In [ ]:
# Save the processed dataset
output_path = '../../data/processed/employee_data_processed.csv'
df_processed.to_csv(output_path, index=False)

print(f"Processed data saved to: {output_path}")
print(f"Final dataset shape: {df_processed.shape}")
print(f"\nProcessed columns: {list(df_processed.columns)}")

## 8. Data Processing Summary

In [ ]:
print("\n" + "="*80)
print("DATA PROCESSING SUMMARY")
print("="*80)
print(f"Original dataset shape: {df.shape}")
print(f"Processed dataset shape: {df_processed.shape}")
print(f"\nData Quality Improvements:")
print(f"  - Duplicates removed: {duplicates}")
print(f"  - Missing values handled: {len(missing_data) if len(missing_data) > 0 else 0} columns")
print(f"  - New features created: {df_processed.shape[1] - df.shape[1]}")
print(f"  - Outliers identified: {len(outlier_summary) if outlier_summary else 0} columns")
print(f"\nData is ready for exploratory analysis and modeling!")
print("="*80)